In [1]:
import torch, torch.optim as optim, numpy as np, gc, warnings
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from IPython.utils import io
from reader import prepare_qsm_dataset
from loader import get_slice_level_data
from util import seed_everything, mask_crop as mask_crop_fn, mean_std_str, compute_comprehensive_metrics, compare_auc_significance
from train import calibrate_balanced
from validate import ClinicalTransformer_cv, ResidualSpectralViT_cv
from networks import FocalLoss, PassThrough, ClinicalTransformer, SpectralViT, SpatialViT, ResidualSpectral, ResidualSpatial

warnings.filterwarnings("ignore", category=UserWarning, message="Default upsampling behavior")

# Configuration
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
N_FOLDS, EPOCHS, JITTER_STD, IMG_AUG_STD, TARGET_DIM = 5, 100, 0.1, 0.05, 128
seed_everything(0)

# Load datasets
dataset_train = prepare_qsm_dataset('MSW', '/media/mts_dbs/dbs/all/nii/qsm_115/im', '/media/mts_dbs/dbs/all/nii/seg_ps/', '/data/Ali/RadDBS-QSM/data/docs/dbs_03292024.csv', './msw_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)
dataset_test = prepare_qsm_dataset('CHH', '/media/mts_dbs/chh/nii/qsm/', '/media/mts_dbs/chh/roi/', '/media/mts_dbs/chh/xlsx/chh_subjects_table1_20240729.csv', './chh_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)

# Extract Arrays
X_full_slices, X_full_clin, y_full_slices, full_subj_map = get_slice_level_data(dataset_train, TARGET_DIM, include_unlabeled=True)
labeled_mask = (y_full_slices != -1)
X_tr_slices, X_tr_clin_raw, y_tr_slices, tr_subj_map = X_full_slices[labeled_mask], X_full_clin[labeled_mask], y_full_slices[labeled_mask], full_subj_map[labeled_mask]
X_te_slices, X_te_clin_raw, y_te_slices, te_subj_map = get_slice_level_data(dataset_test, TARGET_DIM)

# Harmonization & Weights
scaler_train = StandardScaler()
X_tr_clin = scaler_train.fit_transform(X_tr_clin_raw)
scaler_test = StandardScaler()
X_te_clin = scaler_test.fit(X_te_clin_raw[((X_te_clin_raw.shape[0])//4):,:]).transform(X_te_clin_raw)
NEG_WEIGHT = float(sum(y_tr_slices == 1) // sum(y_tr_slices == 0))
GAMMA = NEG_WEIGHT

# Hyperparameter selection
unique_subjs = np.unique(tr_subj_map)
y_unique = np.array([y_tr_slices[tr_subj_map == s][0] for s in unique_subjs])
outer_skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
criterion = FocalLoss(gamma=GAMMA)

SELECTED_POS_WEIGHT = ClinicalTransformer_cv(model_class=ClinicalTransformer, model_kwargs={'n_inputs': X_tr_clin.shape[1]}, pos_weight_grid=[0.1, 0.25, 0.5, 0.75, 1.0], outer_skf=outer_skf, unique_subjs=unique_subjs, y_unique=y_unique, tr_subj_map=tr_subj_map, X_tr_clin=X_tr_clin, y_tr_slices=y_tr_slices, NEG_WEIGHT=NEG_WEIGHT, criterion=criterion, JITTER_STD=JITTER_STD, EPOCHS=EPOCHS, device=device)

model_classes = {'clinical': ClinicalTransformer, 'vit': SpectralViT, 'wrapper': ResidualSpectral}
model_kwargs = {'vit': {'n_heads': 1, 'n_layers': 1, 'embed_dim': 32, 'use_input_proj': False, 'use_pos_embed': False, 'use_layer_norm': False, 'pooling': 'flatten'}}

N_PCA_COMPONENTS = ResidualSpectralViT_cv(model_classes=model_classes, model_kwargs=model_kwargs, pca_components_grid=[16, 32, 64, 128], outer_skf=outer_skf, unique_subjs=unique_subjs, y_unique=y_unique, tr_subj_map=tr_subj_map, X_tr_slices=X_tr_slices, X_tr_clin=X_tr_clin, y_tr_slices=y_tr_slices, NEG_WEIGHT=NEG_WEIGHT, SELECTED_POS_WEIGHT=SELECTED_POS_WEIGHT, criterion=criterion, JITTER_STD=JITTER_STD, EPOCHS=EPOCHS, device=device)

# Cross-validation
print("Training...")
clinical_fold_weights = []
eval_skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
all_cv_probs_ct, all_cv_probs_sp, all_cv_probs_va, all_cv_labels = [], [], [], []
fold_preds_ct, fold_preds_sp, fold_preds_spatial = [], [], []
fold_thresholds_ct, fold_thresholds_sp, fold_thresholds_va = [], [], []

for fold_idx, (train_subj_idx, val_subj_idx) in enumerate(eval_skf.split(unique_subjs, y_unique)):
    print(f"Fold {fold_idx + 1}/{N_FOLDS}")
    train_mask, val_mask = np.isin(tr_subj_map, unique_subjs[train_subj_idx]), np.isin(tr_subj_map, unique_subjs[val_subj_idx])
    
    f_img_scaler = StandardScaler().fit(X_tr_slices[train_mask])
    f_pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0, whiten=True).fit(f_img_scaler.transform(X_tr_slices[train_mask]))
    
    X_train_pca, X_train_clin, X_train_img, y_train = f_pca.transform(f_img_scaler.transform(X_tr_slices[train_mask])), X_tr_clin[train_mask], X_tr_slices[train_mask], y_tr_slices[train_mask]
    X_val_pca, X_val_clin, X_val_img, y_val = f_pca.transform(f_img_scaler.transform(X_tr_slices[val_mask])), X_tr_clin[val_mask], X_tr_slices[val_mask], y_tr_slices[val_mask]
    X_test_pca, X_test_clin, X_test_img = f_pca.transform(f_img_scaler.transform(X_te_slices)), X_te_clin, X_te_slices
    
    X_train_clin_t, X_train_pca_t = torch.tensor(X_train_clin, dtype=torch.float32).to(device), torch.tensor(X_train_pca, dtype=torch.float32).to(device)
    X_train_img_t, y_train_t = torch.tensor(X_train_img).view(-1, 1, 128, 128).float().to(device), torch.tensor(y_train, dtype=torch.float32).to(device)
    w = torch.where(y_train_t == 0, torch.tensor(NEG_WEIGHT, device=device), torch.tensor(SELECTED_POS_WEIGHT, device=device))

    m_ct = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
    opt_ct = optim.Adam(m_ct.parameters(), lr=1e-4)
    for _ in range(EPOCHS):
        m_ct.train(); opt_ct.zero_grad()
        loss = criterion(m_ct(X_train_clin_t + torch.randn_like(X_train_clin_t) * JITTER_STD), y_train_t, weight=w)
        loss.backward(); opt_ct.step()
    
    m_ct.eval(); [p.requires_grad_(False) for p in m_ct.parameters()]
    clinical_fold_weights.append(m_ct.state_dict())

    m_sp = ResidualSpectral(m_ct, SpectralViT(n_inputs=N_PCA_COMPONENTS, n_heads=1, n_layers=1, embed_dim=32, use_input_proj=False, use_pos_embed=False, use_layer_norm=False, pooling='flatten')).to(device)
    m_va = ResidualSpatial(m_ct, SpatialViT(size=128, patch_size=16, embed_dim=32, n_heads=1, n_layers=1, dropout=0.1, is_2d=True, use_cls_token=False, use_layer_norm=False)).to(device)
    
    opt_va = optim.Adam(m_va.m_res.parameters(), lr=1e-4)
    for _ in range(EPOCHS):
        m_va.train(); opt_va.zero_grad()
        loss = criterion(m_va(X_train_img_t + torch.randn_like(X_train_img_t) * IMG_AUG_STD, X_train_clin_t), y_train_t, weight=w)
        loss.backward(); opt_va.step()

    with torch.no_grad():
        v_probs_ct, v_probs_sp, v_probs_va, v_labels = [], [], [], []
        for s in unique_subjs[val_subj_idx]:
            m = (tr_subj_map[val_mask] == s); v_labels.append(y_val[m][0])
            v_probs_ct.append(m_ct(torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())
            v_probs_sp.append(m_sp(torch.tensor(X_val_pca[m], dtype=torch.float32).to(device), torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())
            v_probs_va.append(m_va(torch.tensor(X_val_img[m]).view(-1, 1, 128, 128).float().to(device), torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())

        fold_thresholds_ct.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_ct), np.array(v_labels), 'cpu'))
        fold_thresholds_sp.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_sp), np.array(v_labels), 'cpu'))
        fold_thresholds_va.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_va), np.array(v_labels), 'cpu'))
        all_cv_probs_ct.extend(v_probs_ct); all_cv_probs_sp.extend(v_probs_sp); all_cv_probs_va.extend(v_probs_va); all_cv_labels.extend(v_labels)

        t_probs_ct, t_probs_sp, t_probs_va = [], [], []
        for s in np.unique(te_subj_map):
            m = (te_subj_map == s)
            t_probs_ct.append(m_ct(torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
            t_probs_sp.append(m_sp(torch.tensor(X_test_pca[m], dtype=torch.float32).to(device), torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
            t_probs_va.append(m_va(torch.tensor(X_test_img[m]).view(-1, 1, 128, 128).float().to(device), torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
        fold_preds_ct.append(t_probs_ct); fold_preds_sp.append(t_probs_sp); fold_preds_spatial.append(t_probs_va)

th_ct, th_sp, th_va = np.mean(fold_thresholds_ct), np.mean(fold_thresholds_sp), np.mean(fold_thresholds_va)
y_test_labels = np.array([y_te_slices[te_subj_map == s][0] for s in np.unique(te_subj_map)])
cv_probs = {"Clinical": all_cv_probs_ct, "Spectral": all_cv_probs_sp, "Spatial": all_cv_probs_va}
test_probs = {"Clinical": np.mean(fold_preds_ct, axis=0), "Spectral": np.mean(fold_preds_sp, axis=0), "Spatial": np.mean(fold_preds_spatial, axis=0)}
thresholds = {"Clinical": th_ct, "Spectral": th_sp, "Spatial": th_va}


Preparing MSW dataset (Forced 6-dim alignment) 
Pre-flight check: Validating MSW CSV mapping...
--- MSW CSV RAW MEANS ---
  > Age     : 62.13
  > Sex     : 0.26
  > Dur     : 8.47
  > LEDD    : 989.80
  > Off-Pre : 45.91
  > On-Pre  : 19.60

Final MSW Breakdown:
 - Unique Subjects on Disk: 111
 - Labeled Responders (1): 61
 - Labeled Non-Responders (0): 5
 - Unlabeled subjects (-1): 45
 - Verified Realized Means (Matched Data Only):
    > Age     : 63.08
    > Sex     : 0.26
    > Dur     : 8.44
    > LEDD    : 1003.95
    > Off-Pre : 45.62
    > On-Pre  : 19.55

❌ FULL MISSING LIST (45 subjects):
  [3, 4, 5, 8, 12, 13, 14, 17, 18, 21, 22, 24, 25, 27, 28, 31, 32, 34, 35, 37, 39, 40, 41, 42, 49, 50, 52, 54, 57, 61, 65, 67, 74, 76, 81, 82, 84, 88, 89, 94, 99, 101, 104, 105, 116]
Loaded cache with 7790 slices.

Preparing CHH dataset (Forced 6-dim alignment) 
Pre-flight check: Validating CHH CSV mapping...
--- CHH CSV RAW MEANS ---
  > Age     : 63.13
  > Sex     : 0.46
  > Dur     : 8.54

In [2]:
from torch.cuda.amp import GradScaler, autocast
from networks import AttentionUNet

# Flush and initialize
all_cv_probs_un, fold_preds_un, fold_ths_un, scaler_amp = [], [], [], GradScaler()

for fold, (t_subj_idx, v_subj_idx) in enumerate(eval_skf.split(unique_subjs, y_unique)):
    print(f"Fold {fold+1} Attention U-Net: ", end='', flush=True)
    
    # 1. Setup Data & Clinical Model (Unchanged)
    t_mask = np.isin(tr_subj_map, unique_subjs[t_subj_idx])
    scaler_c = StandardScaler().fit(X_tr_clin[t_mask])
    m_ct_fold = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
    m_ct_fold.load_state_dict(clinical_fold_weights[fold]); m_ct_fold.eval()
    
    with torch.no_grad():
        xt_c_gpu = torch.tensor(scaler_c.transform(X_tr_clin[t_mask]), dtype=torch.float32).to(device)
        t_clin_logits = m_ct_fold(xt_c_gpu, return_logit=True).detach()

    xt_i_gpu = torch.tensor(X_tr_slices[t_mask]).view(-1, 1, 128, 128).float().to(device)
    yt_gpu = torch.tensor(y_tr_slices[t_mask], dtype=torch.float32).to(device)
    pos_idx, neg_idx = torch.where(yt_gpu == 1)[0], torch.where(yt_gpu == 0)[0]
    
    batch_size, half_batch = 64, 32
    num_batches = len(pos_idx) // half_batch

    # 2. Replicate FastUNet Call Exactly
    m_un_res = AttentionUNet(
        in_channels=1, 
        base_channels=16, # Creates 16->32->64 pipeline
        fast_mode=True    # Replicates pooling and SpatialAttention (1+att) logic
    ).to(device)
    optimizer = optim.Adam(m_un_res.parameters(), lr=5e-4)
   
    # 3. Training Loop
    for epoch in range(EPOCHS):
        m_un_res.train()
        shuffled_neg = neg_idx[torch.randperm(len(neg_idx))]
        
        for i in range(num_batches):
            n_ids = shuffled_neg[i*half_batch : (i+1)*half_batch]
            p_ids = pos_idx[torch.randint(0, len(pos_idx), (half_batch,))]
            b_idx = torch.cat([p_ids, n_ids])
            
            optimizer.zero_grad(set_to_none=True) 
            with autocast():
                # Correct logit summation
                res_logit = m_un_res(xt_i_gpu[b_idx] + torch.randn_like(xt_i_gpu[b_idx]) * IMG_AUG_STD)
                joint_logit = res_logit + t_clin_logits[b_idx]
                
                bce_loss = F.binary_cross_entropy_with_logits(joint_logit, yt_gpu[b_idx], reduction='none')
                w_batch = torch.where(yt_gpu[b_idx] == 0, torch.tensor(NEG_WEIGHT, device=device), torch.tensor(SELECTED_POS_WEIGHT, device=device))
                loss = (w_batch * (1 - torch.exp(-bce_loss))**GAMMA * bce_loss).mean()
            
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()
            
        if (epoch + 1) % 25 == 0: print(f'{epoch+1}..', end='', flush=True)
    print('Done.')

    # 4. Inference (Unchanged)
    m_un_res.eval()
    with torch.no_grad():
        v_un, v_lbls = [], []
        for s_id in unique_subjs[v_subj_idx]:
            sm = (tr_subj_map == s_id)
            c_v = torch.tensor(scaler_c.transform(X_tr_clin[sm]), dtype=torch.float32).to(device)
            i_v = torch.tensor(X_tr_slices[sm]).view(-1, 1, 128, 128).float().to(device)
            l_c = m_ct_fold(c_v, return_logit=True)
            v_un.append(torch.sigmoid(l_c + m_un_res(i_v)).mean().item())
            v_lbls.append(y_tr_slices[sm][0])
            
        fold_ths_un.append(calibrate_balanced(PassThrough(), None, np.array(v_un), np.array(v_lbls), 'cpu'))
        all_cv_probs_un.extend(v_un)

        t_un = []
        for s_id in np.unique(te_subj_map):
            sm = (te_subj_map == s_id)
            c_t = torch.tensor(scaler_c.transform(X_te_clin[sm]), dtype=torch.float32).to(device)
            i_t = torch.tensor(X_te_slices[sm]).view(-1, 1, 128, 128).float().to(device)
            l_c_t = m_ct_fold(c_t, return_logit=True)
            t_un.append(torch.sigmoid(l_c_t + m_un_res(i_t)).mean().item())
        fold_preds_un.append(t_un)
    torch.cuda.empty_cache()

# Final Results
cv_probs["Attention U-Net"] = all_cv_probs_un
test_probs["Attention U-Net"] = np.mean(fold_preds_un, axis=0)
thresholds["Attention U-Net"] = np.mean(fold_ths_un)

Fold 1 Attention U-Net: 

25..50..75..100..Done.
Fold 2 Attention U-Net: 25..50..75..100..Done.
Fold 3 Attention U-Net: 25..50..75..100..Done.
Fold 4 Attention U-Net: 25..50..75..100..Done.
Fold 5 Attention U-Net: 25..50..75..100..Done.


In [3]:
# Fold-wise predictions
external_fold_preds_all = {
    "Clinical": fold_preds_ct,
    "PCA+LR": [], #fold_preds_lr,
    "PCA+MLP": [], #fold_preds_mlp,
    "Spectral": fold_preds_sp,
    "Spatial": fold_preds_spatial,
    "Attention U-Net": fold_preds_un
}


def get_cv_fold_metrics(name):
    probs = np.array(cv_probs[name])
    labels = np.array(all_cv_labels)
    fold_indices = np.array_split(np.arange(len(labels)), 5)
    
    fold_results = []
    for idxs in fold_indices:
        m = compute_comprehensive_metrics(labels[idxs], probs[idxs], thresholds[name])
        fold_results.append(m)
    return fold_results

def print_table(title, labels, prob_dict, thresh_dict, fold_data_dict, is_cv=False):
    print(f"\n{title}")
    header = f"{'Model':<16} | {'AUC':<18} | {'B-Acc':<18} | {'Spec':<18} | {'F1':<18}"
    print("-" * len(header))
    print(header)
    print("-" * len(header))

    models = ["Clinical", "PCA+LR", "PCA+MLP", "Spectral", "Spatial", "Attention U-Net"]

    for name in models:
        if name not in prob_dict: continue
        m_mean = compute_comprehensive_metrics(np.array(labels), np.array(prob_dict[name]), thresh_dict[name])
        if is_cv:
            f_metrics = get_cv_fold_metrics(name)
        else:
            f_metrics = []
            for f_idx in range(len(fold_data_dict[name])):
                f_metrics.append(compute_comprehensive_metrics(
                    np.array(labels), 
                    np.array(fold_data_dict[name][f_idx]), 
                    thresh_dict[name]
                ))
        
        # Calculate 95% Confidence Intervals using t-distribution across folds (n=5, df=4)
        n_folds = len(f_metrics)
        ci_mult = 2.776 / np.sqrt(n_folds) # t_{0.025, 4} = 2.776
        
        ci_intervals = {}
        for k in m_mean.keys():
            values = [f[k] for f in f_metrics]
            mean_val = np.mean(values)
            std_val = np.std(values, ddof=1) if n_folds > 1 else 0.0
            ci_half = ci_mult * (std_val / np.sqrt(n_folds)) if n_folds > 1 else 0.0
            ci_intervals[k] = (mean_val, ci_half)

        def fmt(key):
            mean_val, ci_half = ci_intervals[key]
            return f"{mean_val:.3f}±{ci_half:.3f}"

        print(f"{name:<16} | {fmt('AUC'):<18} | {fmt('B-Acc'):<18} | {fmt('Spec'):<18} | {fmt('F1'):<18}")

    # Significance test
    print("\nStatistical Significance (Incremental value over Clinical):")
    for name in models[1:]:
        if name in prob_dict:
            diff, p = compare_auc_significance(np.array(labels), np.array(prob_dict["Clinical"]), np.array(prob_dict[name]))
            sig = "*" if p < 0.05 else "n.s."
            print(f"  {name:<16}: ΔAUC {diff:+.3f}, p={p:.4f} ({sig})")

# Internal validation
print_table(
    "INTERNAL VALIDATION (5-Fold CV: Mean ± 95% CI)", 
    all_cv_labels, cv_probs, thresholds, None, is_cv=True
)

# External test
print_table(
    "EXTERNAL TEST PERFORMANCE (CHH Dataset: Mean ± 95% CI)", 
    y_test_labels, test_probs, thresholds, external_fold_preds_all, is_cv=False
)


INTERNAL VALIDATION (5-Fold CV: Mean ± 95% CI)
----------------------------------------------------------------------------------------------------
Model            | AUC                | B-Acc              | Spec               | F1                
----------------------------------------------------------------------------------------------------
Clinical         | 0.810±0.050        | 0.745±0.043        | 0.794±0.071        | 0.812±0.016       
Spectral         | 0.809±0.039        | 0.734±0.026        | 0.797±0.065        | 0.792±0.038       
Spatial          | 0.816±0.040        | 0.749±0.032        | 0.786±0.050        | 0.823±0.019       
Attention U-Net  | 0.877±0.023        | 0.773±0.035        | 0.814±0.039        | 0.834±0.042       

Statistical Significance (Incremental value over Clinical):
  Spectral        : ΔAUC -0.010, p=0.2298 (n.s.)
  Spatial         : ΔAUC +0.004, p=0.6760 (n.s.)
  Attention U-Net : ΔAUC +0.051, p=0.0000 (*)

EXTERNAL TEST PERFORMANCE (CHH Dataset:

In [4]:
# # Head-to-Head Significance: Spectral vs All Models (External Dataset)
# print("\nHead-to-Head Statistical Significance: Spectral ViT vs. Baselines (External Test Set)")
# print("-" * 75)

# if "Spectral" in test_probs:
#     labels_ext = np.array(y_test_labels)
#     probs_spectral = np.array(test_probs["Spectral"])
    
#     # Models to compare against Spectral
#     comparison_models = ["Clinical", "PCA+LR", "PCA+MLP", "Spatial", "Attention U-Net"]
    
#     header = f"{'Comparison (Spectral vs. Baseline)':<38} | {'ΔAUC':<10} | {'p-value':<10} | {'Sig.':<6}"
#     print(header)
#     print("-" * 75)
    
#     for other_name in comparison_models:
#         if other_name in test_probs and len(test_probs[other_name]) > 0:
#             probs_other = np.array(test_probs[other_name])
            
#             # ΔAUC = AUC(Spectral) - AUC(Other Model)
#             diff, p = compare_auc_significance(labels_ext, probs_other, probs_spectral)
#             sig = "*" if p < 0.05 else "n.s."
            
#             comp_label = f"Spectral vs. {other_name}"
#             print(f"{comp_label:<38} | {diff:+.3f}      | {p:.4f}     | {sig:<6}")
            
#     print("-" * 75)
# else:
#     print("Error: Could not find 'Spectral' key in test_probs.")

In [5]:
# import numpy as np
# import scipy.stats as stats
# from sklearn.metrics import balanced_accuracy_score, roc_auc_score


# def compare_specificity_significance(
#     y_true, prob_base, prob_model, thresh_base, thresh_model
# ):
#   """Compares specificity using McNemar's test with continuity correction

#   on the true negative samples.
#   """
#   y_true = np.array(y_true)
#   prob_base = np.array(prob_base)
#   prob_model = np.array(prob_model)

#   # Isolate negative samples (Specificity = True Negative Rate)
#   neg_mask = y_true == 0

#   # Binarize predictions based on model-specific thresholds
#   pred_base_neg = (prob_base[neg_mask] >= thresh_base).astype(int)
#   pred_model_neg = (prob_model[neg_mask] >= thresh_model).astype(int)

#   # 0 means correctly predicted as negative (True Negative)
#   correct_base = pred_base_neg == 0
#   correct_model = pred_model_neg == 0

#   # McNemar's test components
#   # b: baseline correct, model incorrect
#   b = np.sum(correct_base & ~correct_model)
#   # c: baseline incorrect, model correct
#   c = np.sum(~correct_base & correct_model)

#   diff = np.mean(correct_model) - np.mean(correct_base)

#   if b + c == 0:
#     return diff, 1.0

#   # Chi-squared with continuity correction
#   chi2 = (abs(b - c) - 1.0) ** 2 / (b + c)
#   p_value = stats.chi2.sf(chi2, 1)

#   return diff, p_value


# def compare_balanced_accuracy_significance(
#     y_true, prob_base, prob_model, thresh_base, thresh_model, n_permutations=10000
# ):
#   """Compares balanced accuracy using a paired permutation test on predictions."""
#   y_true = np.array(y_true)
#   prob_base = np.array(prob_base)
#   prob_model = np.array(prob_model)

#   pred_base = (prob_base >= thresh_base).astype(int)
#   pred_model = (prob_model >= thresh_model).astype(int)

#   bacc_base = balanced_accuracy_score(y_true, pred_base)
#   bacc_model = balanced_accuracy_score(y_true, pred_model)
#   observed_diff = bacc_model - bacc_base

#   if observed_diff == 0:
#     return 0.0, 1.0

#   count_extreme = 0
#   n = len(y_true)
#   for _ in range(n_permutations):
#     swap = np.random.binomial(1, 0.5, size=n).astype(bool)
#     sim_base_pred = np.where(swap, pred_model, pred_base)
#     sim_model_pred = np.where(swap, pred_base, pred_model)

#     sim_bacc_base = balanced_accuracy_score(y_true, sim_base_pred)
#     sim_bacc_model = balanced_accuracy_score(y_true, sim_model_pred)
#     sim_diff = sim_bacc_model - sim_bacc_base

#     if abs(sim_diff) >= abs(observed_diff):
#       count_extreme += 1

#   p_value = count_extreme / n_permutations
#   return observed_diff, p_value


# # Head-to-Head Significance: Spectral vs All Models (External Dataset)
# print(
#     "\nHead-to-Head Statistical Significance: Spectral ViT vs. Baselines"
#     " (External Test Set)"
# )
# print("-" * 122)

# if "Spectral" in test_probs and "Spectral" in thresholds:
#   labels_ext = np.array(y_test_labels)
#   probs_spectral = np.array(test_probs["Spectral"])
#   thresh_spectral = thresholds["Spectral"]

#   # Models to compare against Spectral
#   comparison_models = ["Clinical", "PCA+LR", "PCA+MLP", "Spatial", "Attention U-Net"]

#   header = (
#       f"{'Comparison (Spectral vs. Baseline)':<38} | {'ΔAUC':<7} |"
#       f" {'p(AUC)':<7} | {'Sig.':<4} | {'ΔB-Acc':<7} | {'p(B-Acc)':<8} |"
#       f" {'Sig.':<4} | {'ΔSpec':<7} | {'p(Spec)':<7} | {'Sig.':<4}"
#   )
#   print(header)
#   print("-" * 122)

#   for other_name in comparison_models:
#     if (
#         other_name in test_probs
#         and len(test_probs[other_name]) > 0
#         and other_name in thresholds
#     ):
#       probs_other = np.array(test_probs[other_name])
#       thresh_other = thresholds[other_name]

#       # AUC Comparison (DeLong)
#       diff_auc, p_auc = compare_auc_significance(
#           labels_ext, probs_other, probs_spectral
#       )
#       sig_auc = "*" if p_auc < 0.05 else "n.s."

#       # Balanced Accuracy Comparison (Permutation Test)
#       diff_bacc, p_bacc = compare_balanced_accuracy_significance(
#           labels_ext, probs_other, probs_spectral, thresh_other, thresh_spectral
#       )
#       sig_bacc = "*" if p_bacc < 0.05 else "n.s."

#       # Specificity Comparison (McNemar's)
#       diff_spec, p_spec = compare_specificity_significance(
#           labels_ext, probs_other, probs_spectral, thresh_other, thresh_spectral
#       )
#       sig_spec = "*" if p_spec < 0.05 else "n.s."

#       comp_label = f"Spectral vs. {other_name}"
#       print(
#           f"{comp_label:<38} | {diff_auc:+.3f} | {p_auc:.4f} | {sig_auc:<4} |"
#           f" {diff_bacc:+.3f} | {p_bacc:.4f}   | {sig_bacc:<4} |"
#           f" {diff_spec:+.3f} | {p_spec:.4f} | {sig_spec:<4}"
#       )

#   print("-" * 122)
# else:
#   print("Error: Could not find 'Spectral' key in test_probs or thresholds.")

In [6]:
import numpy as np
import scipy.stats as stats
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score


# --- Statistical Significance Helper Functions ---
def delong_roc_variance(ground_truth, predictions):
  """Computes the structural components (V10 and V01) for DeLong's variance."""
  m = sum(ground_truth == 1)
  n = sum(ground_truth == 0)

  pos = predictions[ground_truth == 1]
  neg = predictions[ground_truth == 0]

  v10 = np.array([np.sum(neg < p) + 0.5 * np.sum(neg == p) for p in pos]) / n
  v01 = np.array([np.sum(pos > p) + 0.5 * np.sum(pos == p) for p in neg]) / m

  return v10, v01


def compare_auc_significance(y_true, prob_base, prob_model):
  """Exact implementation of the DeLong test for two correlated ROC curves."""
  y_true = np.array(y_true)
  prob_base = np.array(prob_base)
  prob_model = np.array(prob_model)

  auc_base = roc_auc_score(y_true, prob_base)
  auc_model = roc_auc_score(y_true, prob_model)

  v10_base, v01_base = delong_roc_variance(y_true, prob_base)
  v10_model, v01_model = delong_roc_variance(y_true, prob_model)

  S10 = np.cov(v10_base, v10_model)
  S01 = np.cov(v01_base, v01_model)

  m = sum(y_true == 1)
  n = sum(y_true == 0)

  var_diff = (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / m + (
      S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]
  ) / n

  z = (auc_model - auc_base) / np.sqrt(max(var_diff, 1e-8))
  p_value = 2 * (1 - stats.norm.cdf(abs(z)))

  return auc_model - auc_base, p_value


def compare_specificity_significance(
    y_true, prob_base, prob_model, thresh_base, thresh_model
):
  """Compares specificity using McNemar's test with continuity correction

  on the true negative samples.
  """
  y_true = np.array(y_true)
  prob_base = np.array(prob_base)
  prob_model = np.array(prob_model)

  neg_mask = y_true == 0

  pred_base_neg = (prob_base[neg_mask] >= thresh_base).astype(int)
  pred_model_neg = (prob_model[neg_mask] >= thresh_model).astype(int)

  correct_base = pred_base_neg == 0
  correct_model = pred_model_neg == 0

  b = np.sum(correct_base & ~correct_model)
  c = np.sum(~correct_base & correct_model)

  diff = np.mean(correct_model) - np.mean(correct_base)

  if b + c == 0:
    return diff, 1.0

  chi2 = (abs(b - c) - 1.0) ** 2 / (b + c)
  p_value = stats.chi2.sf(chi2, 1)

  return diff, p_value


def compare_balanced_accuracy_significance(
    y_true, prob_base, prob_model, thresh_base, thresh_model, n_permutations=10000
):
  """Compares balanced accuracy using a paired permutation test on predictions."""
  y_true = np.array(y_true)
  prob_base = np.array(prob_base)
  prob_model = np.array(prob_model)

  pred_base = (prob_base >= thresh_base).astype(int)
  pred_model = (prob_model >= thresh_model).astype(int)

  bacc_base = balanced_accuracy_score(y_true, pred_base)
  bacc_model = balanced_accuracy_score(y_true, pred_model)
  observed_diff = bacc_model - bacc_base

  if observed_diff == 0:
    return 0.0, 1.0

  count_extreme = 0
  n = len(y_true)
  for _ in range(n_permutations):
    swap = np.random.binomial(1, 0.5, size=n).astype(bool)
    sim_base_pred = np.where(swap, pred_model, pred_base)
    sim_model_pred = np.where(swap, pred_base, pred_model)

    sim_bacc_base = balanced_accuracy_score(y_true, sim_base_pred)
    sim_bacc_model = balanced_accuracy_score(y_true, sim_model_pred)
    sim_diff = sim_bacc_model - sim_bacc_base

    if abs(sim_diff) >= abs(observed_diff):
      count_extreme += 1

  p_value = count_extreme / n_permutations
  return observed_diff, p_value


def compare_f1_significance(
    y_true, prob_base, prob_model, thresh_base, thresh_model, n_permutations=10000
):
  """Compares F1 score using a paired permutation test on predictions."""
  y_true = np.array(y_true)
  prob_base = np.array(prob_base)
  prob_model = np.array(prob_model)

  pred_base = (prob_base >= thresh_base).astype(int)
  pred_model = (prob_model >= thresh_model).astype(int)

  f1_base = f1_score(y_true, pred_base, zero_division=0)
  f1_model = f1_score(y_true, pred_model, zero_division=0)
  observed_diff = f1_model - f1_base

  if observed_diff == 0:
    return 0.0, 1.0

  count_extreme = 0
  n = len(y_true)
  for _ in range(n_permutations):
    swap = np.random.binomial(1, 0.5, size=n).astype(bool)
    sim_base_pred = np.where(swap, pred_model, pred_base)
    sim_model_pred = np.where(swap, pred_base, pred_model)

    sim_f1_base = f1_score(y_true, sim_base_pred, zero_division=0)
    sim_f1_model = f1_score(y_true, sim_model_pred, zero_division=0)
    sim_diff = sim_f1_model - sim_f1_base

    if abs(sim_diff) >= abs(observed_diff):
      count_extreme += 1

  p_value = count_extreme / n_permutations
  return observed_diff, p_value


# Fold-wise predictions
external_fold_preds_all = {
    "Clinical": fold_preds_ct,
    "PCA+LR": [],  # fold_preds_lr,
    "PCA+MLP": [],  # fold_preds_mlp,
    "Spectral": fold_preds_sp,
    "Spatial": fold_preds_spatial,
    "Attention U-Net": fold_preds_un,
}


def get_cv_fold_metrics(name):
  probs = np.array(cv_probs[name])
  labels = np.array(all_cv_labels)
  fold_indices = np.array_split(np.arange(len(labels)), 5)

  fold_results = []
  for idxs in fold_indices:
    m = compute_comprehensive_metrics(labels[idxs], probs[idxs], thresholds[name])
    fold_results.append(m)
  return fold_results


# Internal Validation 
print("\nINTERNAL VALIDATION (5-Fold CV: Mean ± 95% CI)")
header_cv = f"{'Model':<16} | {'AUC':<18} | {'B-Acc':<18} | {'Spec':<18} | {'F1':<18}"
print("-" * len(header_cv))
print(header_cv)
print("-" * len(header_cv))

models = ["Clinical", "PCA+LR", "PCA+MLP", "Spectral", "Spatial", "Attention U-Net"]

for name in models:
  if name not in cv_probs:
    continue
  m_mean = compute_comprehensive_metrics(
      np.array(all_cv_labels), np.array(cv_probs[name]), thresholds[name]
  )
  f_metrics = get_cv_fold_metrics(name)

  n_folds = len(f_metrics)
  ci_mult = 2.776 / np.sqrt(n_folds)

  ci_intervals = {}
  for k in m_mean.keys():
    values = [f[k] for f in f_metrics]
    mean_val = np.mean(values)
    std_val = np.std(values, ddof=1) if n_folds > 1 else 0.0
    ci_half = ci_mult * (std_val / np.sqrt(n_folds)) if n_folds > 1 else 0.0
    ci_intervals[k] = (mean_val, ci_half)


  def fmt_cv(key):
    mean_val, ci_half = ci_intervals[key]
    return f"{mean_val:.3f}±{ci_half:.3f}"


  print(
      f"{name:<16} | {fmt_cv('AUC'):<18} | {fmt_cv('B-Acc'):<18} |"
      f" {fmt_cv('Spec'):<18} | {fmt_cv('F1'):<18}"
  )

print("\nStatistical Significance (Incremental value over Clinical):")
for name in models[1:]:
  if name in cv_probs:
    diff, p = compare_auc_significance(
        np.array(all_cv_labels),
        np.array(cv_probs["Clinical"]),
        np.array(cv_probs[name]),
    )
    sig = "*" if p < 0.05 else "-"
    print(f"  {name:<16}: ΔAUC {diff:+.3f}, p={p:.4f} ({sig})")


# External Test Performance
print("\n")
print("=" * 148)
print(
    "External test performance & significance versus Spectral ViT"
)
print("=" * 148)

if "Spectral" in test_probs and "Spectral" in thresholds:
  labels_ext = np.array(y_test_labels)
  probs_spectral = np.array(test_probs["Spectral"])
  thresh_spectral = thresholds["Spectral"]

  header_ext = (
      f"{'Model':<16} | {'AUC':<16} | {'B-Acc':<16} | {'Spec':<16} |"
      f" {'F1':<16} || {'ΔAUC':<12} | {'ΔB-Acc':<12} | {'ΔSpec':<12} |"
      f" {'ΔF1':<12}"
  )
  print(header_ext)
  print("-" * 148)

  for name in models:
    if name not in test_probs or len(test_probs[name]) == 0:
      continue

    # Compute performance metrics & confidence intervals
    m_mean = compute_comprehensive_metrics(
        labels_ext, np.array(test_probs[name]), thresholds[name]
    )
    f_metrics = []
    for f_idx in range(len(external_fold_preds_all[name])):
      f_metrics.append(
          compute_comprehensive_metrics(
              labels_ext,
              np.array(external_fold_preds_all[name][f_idx]),
              thresholds[name],
          )
      )

    n_folds = len(f_metrics)
    ci_mult = 2.776 / np.sqrt(n_folds) if n_folds > 0 else 0.0

    ci_intervals = {}
    for k in m_mean.keys():
      values = [f[k] for f in f_metrics]
      mean_val = np.mean(values) if values else m_mean[k]
      std_val = np.std(values, ddof=1) if n_folds > 1 else 0.0
      ci_half = ci_mult * (std_val / np.sqrt(n_folds)) if n_folds > 1 else 0.0
      ci_intervals[k] = (mean_val, ci_half)


    def fmt_ext(key):
      mean_val, ci_half = ci_intervals[key]
      return f"{mean_val:.3f}±{ci_half:.3f}"


    perf_str = (
        f"{name:<16} | {fmt_ext('AUC'):<16} | {fmt_ext('B-Acc'):<16} |"
        f" {fmt_ext('Spec'):<16} | {fmt_ext('F1'):<16}"
    )

    # Compute significance against Spectral ViT
    if name != "Spectral":
      probs_other = np.array(test_probs[name])
      thresh_other = thresholds[name]

      diff_auc, p_auc = compare_auc_significance(
          labels_ext, probs_other, probs_spectral
      )
      sig_auc = "*" if p_auc < 0.05 else "-"

      diff_bacc, p_bacc = compare_balanced_accuracy_significance(
          labels_ext, probs_other, probs_spectral, thresh_other, thresh_spectral
      )
      sig_bacc = "*" if p_bacc < 0.05 else "-"

      diff_spec, p_spec = compare_specificity_significance(
          labels_ext, probs_other, probs_spectral, thresh_other, thresh_spectral
      )
      sig_spec = "*" if p_spec < 0.05 else "-"

      diff_f1, p_f1 = compare_f1_significance(
          labels_ext, probs_other, probs_spectral, thresh_other, thresh_spectral
      )
      sig_f1 = "*" if p_f1 < 0.05 else "-"

      sig_str = (
          f" || {diff_auc:+.3f}({sig_auc})   | {diff_bacc:+.3f}({sig_bacc})"
          f"   | {diff_spec:+.3f}({sig_spec})   | {diff_f1:+.3f}({sig_f1})"
      )
    else:
      sig_str = (
          f" || {'-':<12}| {'-':<12}| {'-':<12}| {'-':<12}"
      )

    print(perf_str + sig_str)

  print("-" * 148)
else:
  print("Error: Could not find 'Spectral' key in test_probs or thresholds.")


INTERNAL VALIDATION (5-Fold CV: Mean ± 95% CI)
----------------------------------------------------------------------------------------------------
Model            | AUC                | B-Acc              | Spec               | F1                
----------------------------------------------------------------------------------------------------


Clinical         | 0.810±0.050        | 0.745±0.043        | 0.794±0.071        | 0.812±0.016       
Spectral         | 0.809±0.039        | 0.734±0.026        | 0.797±0.065        | 0.792±0.038       
Spatial          | 0.816±0.040        | 0.749±0.032        | 0.786±0.050        | 0.823±0.019       
Attention U-Net  | 0.877±0.023        | 0.773±0.035        | 0.814±0.039        | 0.834±0.042       

Statistical Significance (Incremental value over Clinical):
  Spectral        : ΔAUC -0.010, p=0.2298 (-)
  Spatial         : ΔAUC +0.004, p=0.6760 (-)
  Attention U-Net : ΔAUC +0.051, p=0.0000 (*)


External test performance & significance versus Spectral ViT
Model            | AUC              | B-Acc            | Spec             | F1               || ΔAUC         | ΔB-Acc       | ΔSpec        | ΔF1         
----------------------------------------------------------------------------------------------------------------------------------------------------
Clinical         | 0.769±0.028 